In [4]:
import torch
import torch.nn as nn
import numpy

In [ ]:
class SingleEncoderBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d()
        self.batch_norm1 = nn.BatchNorm2d()
        self.conv2 = nn.Conv2d()
        self.batch_norm2 = nn.BatchNorm2d()
        self.activation_fun = nn.ReLU()
        self.pool = nn.MaxPool2d()
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.batch_norm1(x)
        x = self.activation_fun(x)

        x = self.conv2(x)
        x = self.batch_norm2(x)
        x = self.activation_fun(x)

        p = self.pool(x)
        return x, p

In [ ]:
class SingleDecoderBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.up_conv = nn.ConvTranspose2d()

        self.conv1 = nn.Conv2d()
        self.batch_norm1 = nn.BatchNorm2d()
        self.conv2 = nn.Conv2d()
        self.batch_norm2 = nn.BatchNorm2d()
        self.activation_fun = nn.ReLU()
    
    def forward(self, input, skip_layer_input):
        input = self.up_conv(input)
        merged = torch.cat([input, skip_layer_input], axis=1)

        x = self.conv1(merged)
        x = self.batch_norm1(x)
        x = self.activation_fun(x)

        x = self.conv2(x)
        x = self.batch_norm2(x)
        x = self.activation_fun(x)

        return x



In [ ]:
class Full2DUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder1 = SingleEncoderBlock()
        self.encoder2 = SingleEncoderBlock()
        self.encoder3 = SingleEncoderBlock()
        self.encoder4 = SingleEncoderBlock()

        self.bottleneck = nn.Sequential(
            nn.Conv2d(),
            nn.BatchNorm2d(),
            nn.Conv2d(),
            nn.BatchNorm2d(),
            nn.ReLU()
        )

        self.decoder1 = SingleDecoderBlock()
        self.decoder2 = SingleDecoderBlock()
        self.decoder3 = SingleDecoderBlock()
        self.decoder4 = SingleDecoderBlock()
        
        self.output_layer = nn.Conv2D()

    def forward(self,input):
        x1, p1 = self.encoder1(input)
        x2, p2 = self.encoder2(p1)
        x3, p3 = self.encoder3(p2)
        x4, p4 = self.encoder4(p3)

        b = self.bottleneck(p4)

        d1 = self.decoder1(b, x4)
        d2 = self.decoder2(d1, x3)
        d3 = self.decoder3(d2, x2)
        d4 = self.decoder4(d3, x1)

        output = self.output_layer(d4)

        return output